# Several behaviors on one frozen model

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/pfekin/LARA/blob/main/examples/behaviors/granite-4.0-1b/test.ipynb)

This notebook downloads behaviors trained by `train` and shows what
happens when they share one frozen model. Nothing is trained here, so it runs in
a few minutes.

There is nothing to compare this against. No other method keeps several
behaviors resident on a frozen base and routes between them, so this is a
systems demonstration rather than a benchmark. It reports four things:

- **each behavior alone**, at several strengths, with strength 0 as the check
  that the base is untouched
- **ratio**: how much of each behavior survives once it shares a router
- **routing**: which behavior each domain's tokens actually reach
- **footprint**: one base plus N behaviors against N separate models

Both routing modes are shown. Hard sends each token to a single behavior. Soft
blends them, which lets a style behavior apply on the same token as a domain
one. Hard routing cannot do that by construction.

## Configuration

In [16]:
!pip install -q git+https://github.com/pfekin/LARA.git
!pip install -q transformers datasets accelerate

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [17]:
import gc, json, math, os, random
import numpy as np
import torch
import torch.nn.functional as F
from transformers import AutoModelForCausalLM, AutoTokenizer

from lara import LARA, Bank

NL = chr(10)

# ── the model ────────────────────────────────────────────────────────────────
# Any causal LM. Nothing below assumes a particular one.
BASE = "ibm-granite/granite-4.0-1b"

# ── the behaviors ────────────────────────────────────────────────────────────
# Comment entries out to train fewer. Everything adapts to the list.
# `layers` and `rank` override the defaults per behavior: preference training
# needs far less placement than fine-tuning, so `polite` uses a single module.
BEHAVIORS = [
    {"name": "code",    "task": "ce",  "hf": "sahil2801/CodeAlpaca-20k", "split": "train",
     "system": "You are an expert programmer.",
     "map": lambda r: (r.get("instruction", ""), r.get("output", ""))},
    {"name": "math",    "task": "ce",  "hf": "meta-math/MetaMathQA", "split": "train",
     "system": "You are a math tutor.",
     "map": lambda r: (r.get("query", ""), r.get("response", ""))},
    {"name": "medical", "task": "ce",  "hf": "lavita/ChatDoctor-HealthCareMagic-100k",
     "split": "train", "system": "You are a medical expert.",
     "map": lambda r: (r.get("instruction", ""), r.get("output", ""))},
    {"name": "summary", "task": "ce",  "hf": "knkarthick/dialogsum", "split": "train",
     "system": "You are a summarization assistant.",
     "map": lambda r: ("Summarize this conversation:" + NL + r.get("dialogue", ""),
                       r.get("summary", ""))},
    {"name": "polite",  "task": "dpo", "source": "synthetic", "layers": 1, "max_len": 256,
     "system": "You are a helpful assistant."},
]

# ── where behaviors live ─────────────────────────────────────────────────────
# One repo, one folder per base model, because a behavior only loads onto the
# base it was trained against.
HF_USER       = "pfekin"
BEHAVIOR_REPO = f"{HF_USER}/lara-behaviors"
MODEL_SLUG    = BASE.split("/")[-1].lower()

# ── sizes ────────────────────────────────────────────────────────────────────
BF16   = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
DTYPE  = torch.bfloat16 if BF16 else torch.float16
GAMMAS = (0.0, 0.5, 1.0, 1.5)     # 0.0 is the untouched base; past 1.0 to find the peak

MAX_LEN, N_TRAIN, N_EVAL = 384, 700, 96
EVAL_BS                  = 8      # lower it if evaluation runs out of memory
CE_STEPS, CE_LR          = 700, 2e-4
DPO_PAIRS, DPO_EVAL      = 4000, 600
DPO_STEPS, DPO_LR        = 2400, 5e-5
LAYERS, RANK, ALPHA      = 6, 128, 128
LENGTH_NORM = True
# Beta must match the scale of the logprobs. Summed logprobs differ by tens of
# nats, so 0.1 fits. Length-normalised ones differ by hundredths, and the same
# beta would leave the margin at zero.
BETA        = 2.0 if LENGTH_NORM else 0.1
NLL_LAMBDA  = 0.2     # anchors the chosen answer so the margin is not widened
                      # by pushing down on tokens both answers share

if torch.cuda.is_available():
    gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"{torch.cuda.get_device_name(0)}  {gb:.0f} GB  bf16={'yes' if BF16 else 'no'}")
else:
    print("no GPU: pick a GPU runtime")
print(f"base   {BASE}")
print(f"repo   {BEHAVIOR_REPO}/{MODEL_SLUG}")
print(f"{len(BEHAVIORS)} behaviors: {[b['name'] for b in BEHAVIORS]}")

NVIDIA L4  24 GB  bf16=yes
base   ibm-granite/granite-4.0-1b
repo   pfekin/lara-behaviors/granite-4.0-1b
5 behaviors: ['code', 'math', 'medical', 'summary', 'polite']


In [18]:
from huggingface_hub import hf_hub_download, list_repo_files
from safetensors import safe_open

def weight_family(repo):
    """Low-bit models are often published as their small weights written into a
    wider container. One bit per weight means a block of 128 holds two distinct
    values; ternary holds three; an ordinary model holds 128."""
    shards = sorted(f for f in list_repo_files(repo) if f.endswith(".safetensors"))
    if not shards:
        return "unknown", 16.0
    # torch rather than numpy: numpy has no bfloat16, which many models ship in.
    with safe_open(hf_hub_download(repo, shards[0]), framework="pt") as f:
        key = next((k for k in f.keys() if k.endswith("gate_proj.weight")), None)
        if key is None:
            return "unknown", 16.0
        row = f.get_tensor(key)[0].float().numpy()
    counts = {len(np.unique(row[i*128:(i+1)*128])) for i in range(16)}
    zeros = float((row[:2048] == 0.0).mean())
    if counts <= {1, 2} and zeros == 0:
        return "1-bit", 1.125
    if counts <= {1, 2, 3} and zeros > 0:
        return "ternary", 2.0
    return "full precision", 16.0


FAMILY, BITS = weight_family(BASE)
print(f"{BASE}: {FAMILY}  (~{BITS} bits per weight as shipped)")

ibm-granite/granite-4.0-1b: unknown  (~16.0 bits per weight as shipped)


## 1. Fetch the behaviors

They come from the folder `routed_train` wrote, under this model's name.

In [19]:
from huggingface_hub import snapshot_download, repo_exists

if os.path.isdir("behaviors") and all(
        os.path.isdir(f"behaviors/{b['name']}") for b in BEHAVIORS):
    print("using the local behaviors/ folder")
else:
    if not repo_exists(BEHAVIOR_REPO, repo_type="model"):
        raise SystemExit(f"{BEHAVIOR_REPO} does not exist yet. Run routed_train first.")
    snapshot_download(BEHAVIOR_REPO, repo_type="model", local_dir="_hub",
                      allow_patterns=[f"{MODEL_SLUG}/*"])
    src = os.path.join("_hub", MODEL_SLUG)
    if not os.path.isdir(src):
        raise SystemExit(
            f"{BEHAVIOR_REPO} has no folder for {MODEL_SLUG}. Either BASE is not the "
            f"model these behaviors were trained on, or routed_train has not run for it.")
    os.replace(src, "behaviors")
    print(f"downloaded {BEHAVIOR_REPO}/{MODEL_SLUG}")

missing = [b["name"] for b in BEHAVIORS if not os.path.isdir(f"behaviors/{b['name']}")]
assert not missing, f"no folder for {missing}; check BEHAVIORS against what was trained"
for b in BEHAVIORS:
    mb = sum(os.path.getsize(os.path.join(d, f))
             for d, _, fs in os.walk(f"behaviors/{b['name']}") for f in fs) / 1e6
    print(f"  {b['name']:<10}{mb:>7.1f} MB   {b.get('layers', LAYERS)} modules")

using the local behaviors/ folder
  code         12.8 MB   6 modules
  math         12.9 MB   6 modules
  medical      12.9 MB   6 modules
  summary      13.0 MB   6 modules
  polite        2.2 MB   1 modules


## 2. Evaluation data

The same held-out sets the training notebook used, rebuilt from the same seeds.
Nothing here was trained on.

In [20]:
from datasets import load_dataset
import itertools

def chatml(tok, system, instr, answer):
    msgs = ([{"role": "system", "content": system}] if system else []) \
         + [{"role": "user", "content": instr}]
    try:
        p = tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True,
                                    enable_thinking=False)
    except TypeError:
        p = tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    return p, p + answer + tok.eos_token


def load_ce(tok, spec, n_train, n_eval):
    """Streamed, so nothing large is downloaded. Eval is held out from train."""
    rows = list(itertools.islice(
        load_dataset(spec["hf"], split=spec["split"], streaming=True), n_train + n_eval))
    out = []
    for r in rows:
        try:
            instr, ans = spec["map"](r)
        except Exception:
            continue
        if instr and ans and len(str(ans)) > 20:
            out.append(chatml(tok, spec["system"], str(instr)[:1500], str(ans)[:2000]))
    random.Random(0).shuffle(out)
    return out[n_eval:], out[:n_eval]


# ── synthetic preference pairs ───────────────────────────────────────────────
# Same content, same length, differing only in manner: one answer commits, the
# other hedges. Matching the length matters. If the hedged side ran longer, a
# model could score perfectly by counting words and never read the hedging.
TOPIC = ["how a compiler works", "why the sky is blue", "how to boil an egg",
         "what a database index does", "how a bicycle gear works", "why bread rises",
         "how a fuse protects a circuit", "what a checksum is for", "how sound travels",
         "why ice floats", "how a heat pump moves heat", "why metals conduct electricity",
         "what a cache miss costs", "how yeast ferments sugar",
         "why bridges have expansion joints", "how noise cancelling works",
         "what a hash function guarantees", "how sails work upwind",
         "why batteries lose capacity", "how a microphone captures sound"]
BODY = ["it comes down to a few steps that build on each other",
        "the mechanism is simpler than it first appears",
        "two things matter, and the second follows from the first",
        "one process feeds directly into another",
        "there is a cause and an effect, and they are easy to separate",
        "a small difference at the start compounds later on",
        "the energy has to go somewhere, and that is the whole trick",
        "it is a trade between speed and accuracy",
        "the shape does most of the work, not the material",
        "timing matters more than force here"]
OPENERS = [  # (direct, hedged) -- identical word counts
    ("In short, plainly", "I think, maybe"),
    ("Briefly, and clearly", "Possibly, though unsure"),
    ("Put simply, definitely", "I guess, perhaps"),
    ("The short answer is", "It might just be"),
    ("Essentially, quite clearly", "Arguably, though possibly"),
    ("At its core, certainly", "I am unsure, but"),
    ("Simply put, without doubt", "It could be perhaps"),
    ("Fundamentally, and firmly", "Conceivably, though tentatively"),
    ("Clearly, and directly", "Maybe, though doubtfully"),
    ("Precisely, and simply", "Seemingly, though vaguely"),
]
CLOSERS = [  # (confident, waffling) -- identical word counts
    ("That is the core of it.", "Though I am not really certain."),
    ("That covers the main idea.", "But do check somewhere else."),
    ("That is what makes it work.", "At least, I believe so mostly."),
    ("That is the essential part.", "Or something close to it."),
    ("That is the whole mechanism.", "More or less, I suppose."),
    ("That is the key point.", "Perhaps, though I forget now."),
    ("That explains the behaviour.", "Roughly, if memory serves."),
    ("That is the short version.", "Or thereabouts, I would gather."),
    ("That is all it amounts to.", "Though the sources may well differ."),
    ("That is the underlying reason.", "Assuming I recall this correctly."),
]
assert all(len(a.split()) == len(b.split()) for a, b in OPENERS + CLOSERS), \
    "openers and closers must match word for word, or length leaks into the signal"


def synthetic_pairs(tok, spec, n, held_out=False):
    """Training uses the first 70% of each phrase list, evaluation the rest.
    Without that split the eval pairs reuse phrasings seen thousands of times,
    every margin clears zero by a mile, and accuracy measures nothing."""
    def split(xs):
        k = int(len(xs) * 0.7)
        return xs[k:] if held_out else xs[:k]

    op, cl, tp, bd = split(OPENERS), split(CLOSERS), split(TOPIC), split(BODY)
    rng = random.Random(23 if held_out else 11)
    out = []
    for _ in range(n):
        b = rng.choice(bd)
        od, oh = rng.choice(op)
        cd, ch = rng.choice(cl)
        pr, fc = chatml(tok, spec["system"], f"Explain {rng.choice(tp)}.", f"{od} {b}. {cd}")
        _,  fj = chatml(tok, spec["system"], f"Explain {rng.choice(tp)}.", f"{oh} {b}. {ch}")
        out.append((pr, fc, fj))
    return out


_tok = AutoTokenizer.from_pretrained(BASE)
_tok.pad_token = _tok.pad_token or _tok.eos_token
DATA = {}
for b in BEHAVIORS:
    if b["task"] == "ce":
        tr, ev = load_ce(_tok, b, N_TRAIN, N_EVAL)
        DATA[b["name"]] = {"train": tr, "eval": ev}
        print(f"  {b['name']:<9} {len(tr)} train / {len(ev)} eval")
    else:
        tr = synthetic_pairs(_tok, b, DPO_PAIRS)
        ev = synthetic_pairs(_tok, b, DPO_EVAL, held_out=True)
        DATA[b["name"]] = {"pairs": tr, "pairs_eval": ev}
        se = 1.96 * (0.25 / len(ev)) ** 0.5
        shorter = np.mean([len(c.split()) < len(j.split()) for _, c, j in tr])
        print(f"  {b['name']:<9} {len(tr)} pairs / {len(ev)} eval pairs (+/-{se:.3f} at 95%)")
        print(f"  {'':<9} 'shorter is better' scores {shorter:.3f}; 0.5 means length "
              f"carries no signal")

  code      642 train / 96 eval
  math      699 train / 96 eval
  medical   695 train / 96 eval
  summary   700 train / 96 eval
  polite    4000 pairs / 600 eval pairs (+/-0.040 at 95%)
            'shorter is better' scores 0.346; 0.5 means length carries no signal


## 3. Measurement

In [21]:
def load_base():
    tok = AutoTokenizer.from_pretrained(BASE, clean_up_tokenization_spaces=False)
    tok.pad_token = tok.pad_token or tok.eos_token
    m = AutoModelForCausalLM.from_pretrained(BASE, dtype=DTYPE, device_map="auto")
    return m, tok


def logp(model, ids, n_prompt, norm):
    """Single-sequence version, used by the training loop where a gradient is
    needed. Evaluation uses score_batch instead."""
    lg = model(ids).logits[:, :-1].float()
    lp = torch.log_softmax(lg, -1).gather(-1, ids[:, 1:].unsqueeze(-1)).squeeze(-1)
    lp = lp[:, n_prompt - 1:]
    return (lp.mean() if norm else lp.sum()), -lp.mean()


def enc_pair(tok, prompt, full, max_len=None):
    p = tok(prompt, add_special_tokens=False).input_ids
    f = tok(full, add_special_tokens=False).input_ids[:(max_len or MAX_LEN)]
    return torch.tensor([f]), min(len(p), len(f))


@torch.no_grad()
def score_batch(model, tok, items, max_len=None, bs=None):
    """Completion log probabilities for a list of (prompt, full) pairs.

    Batched. One forward pass per example is the difference between a minute and
    a quarter of an hour once you sweep several strengths."""
    max_len, bs = max_len or MAX_LEN, bs or EVAL_BS
    pad = tok.pad_token_id
    out = []
    for i in range(0, len(items), bs):
        chunk = items[i:i + bs]
        enc = [tok(f, add_special_tokens=False).input_ids[:max_len] for _, f in chunk]
        npr = [min(len(tok(p, add_special_tokens=False).input_ids), len(e) - 1)
               for (p, _), e in zip(chunk, enc)]
        width = max(len(e) for e in enc)
        ids = torch.full((len(enc), width), pad, dtype=torch.long)
        att = torch.zeros((len(enc), width), dtype=torch.long)
        for k, e in enumerate(enc):
            ids[k, :len(e)] = torch.tensor(e)
            att[k, :len(e)] = 1
        ids, att = ids.to(model.device), att.to(model.device)
        lg = model(ids, attention_mask=att).logits[:, :-1].float()
        lp = torch.log_softmax(lg, -1).gather(-1, ids[:, 1:].unsqueeze(-1)).squeeze(-1)
        for k, e in enumerate(enc):
            seg = lp[k, max(npr[k] - 1, 0):len(e) - 1]
            out.append((seg.sum().item(), max(seg.numel(), 1)))
    return out


def ppl(model, tok, texts):
    tot = score_batch(model, tok, texts)
    return math.exp(sum(-s for s, _ in tot) / sum(n for _, n in tot))


def ref_logprobs(model, tok, pairs, max_len=None):
    """The reference model is the base. The modules start at zero, so the model
    with nothing attached IS the reference: no second copy is needed."""
    ch = score_batch(model, tok, [(p, c) for p, c, _ in pairs], max_len)
    rj = score_batch(model, tok, [(p, j) for p, _, j in pairs], max_len)
    norm = (lambda s, n: s / n) if LENGTH_NORM else (lambda s, n: s)
    return [(norm(a, na), norm(b, nb)) for (a, na), (b, nb) in zip(ch, rj)]


def reward(model, tok, pairs, ref, max_len=None):
    """Accuracy thresholds the margin at zero, so it saturates once the
    separation is wide. The margin keeps moving after that, so both are shown."""
    ch = score_batch(model, tok, [(p, c) for p, c, _ in pairs], max_len)
    rj = score_batch(model, tok, [(p, j) for p, _, j in pairs], max_len)
    norm = (lambda s, n: s / n) if LENGTH_NORM else (lambda s, n: s)
    hit, ms = 0.0, []
    for (a, na), (b, nb), (rc, rjj) in zip(ch, rj, ref):
        d = (norm(a, na) - rc) - (norm(b, nb) - rjj)
        if not math.isfinite(d):
            continue
        # At strength 0 the policy IS the reference, so every margin is exactly
        # zero. Counting ties as a half puts that row at chance, where it belongs.
        hit += 1.0 if d > 0 else (0.5 if d == 0 else 0.0)
        ms.append(d)
    n = max(len(ms), 1)
    return hit / n, float(np.mean(ms)) if ms else float("nan")


def measure(model, tok, spec, refs):
    """One number per behavior, in whichever metric fits its objective."""
    if spec["task"] == "ce":
        return {"metric": "PPL", "want": "lower",
                "value": ppl(model, tok, DATA[spec["name"]]["eval"])}
    acc, mar = reward(model, tok, DATA[spec["name"]]["pairs_eval"],
                      refs[spec["name"]], spec.get("max_len"))
    return {"metric": "REWARD", "want": "higher", "value": acc, "margin": mar}


def eval_all(model, tok, refs, gammas, bank=None):
    """Every behavior at every strength, on one loaded model.

    bank=None measures the bare base. Otherwise each behavior is pinned in turn,
    so the model is loaded once rather than once per behavior."""
    out = {}
    for b in BEHAVIORS:
        row = {}
        for g in gammas:
            if bank is None:
                row[g] = measure(model, tok, b, refs)
            else:
                with bank.pin({b["name"]: g}):
                    row[g] = measure(model, tok, b, refs)
        out[b["name"]] = row
        print(f"  {b['name']:<9} " + "  ".join(f"{g}: {row[g]['value']:.3f}"
                                               for g in gammas))
    return out


In [22]:
def solo_table(solo, baseline):
    print("PPL is perplexity: lower is better.")
    print("REWARD is preference accuracy against the base: higher is better, "
          "0.5 is chance.")
    print()
    hdr = f"{'behavior':<10}{'metric':<8}{'want':<7}{'base':>10}"
    print(hdr + "".join(f"{'g=' + str(g):>9}" for g in GAMMAS))
    print("-" * (len(hdr) + 9 * len(GAMMAS)))
    for b in BEHAVIORS:
        n, r = b["name"], solo[b["name"]]
        print(f"{n:<10}{r[GAMMAS[0]]['metric']:<8}{r[GAMMAS[0]]['want']:<7}"
              f"{baseline[n]:>10.3f}" + "".join(f"{r[g]['value']:>9.3f}" for g in GAMMAS))
        if b["task"] == "dpo":
            print(f"{'':<10}{'margin/tok':<8}{'higher':<7}{0.0:>10.3f}"
                  + "".join(f"{r[g]['margin']:>+9.3f}" for g in GAMMAS))
    print()
    for b in BEHAVIORS:
        n, r = b["name"], solo[b["name"]]
        if b["task"] == "ce":
            best = min(GAMMAS, key=lambda g: r[g]["value"])
            chg = (baseline[n] - r[best]["value"]) / baseline[n]
            print(f"  {n:<10} best at strength {best}: {chg:.0%} lower perplexity")
        else:
            # accuracy ties are common once the margin is wide; break on margin
            best = max(GAMMAS, key=lambda g: (r[g]["value"], r[g]["margin"]))
            print(f"  {n:<10} best at strength {best}: "
                  f"accuracy {baseline[n]:.3f} -> {r[best]['value']:.3f}, "
                  f"margin +0.000 -> {r[best]['margin']:+.3f} per token")
    drift = max(abs(solo[b["name"]][0.0]["value"] - baseline[b["name"]])
                / max(baseline[b["name"]], 1e-9) for b in BEHAVIORS)
    print()
    print(f"largest gap between the bare base and strength 0: {drift:.2%}")
    print("strength 0 reproduces the base exactly: the correction is scaled to nothing")

## 4. Each behavior alone

One behavior loaded at a time. Strength 0 should reproduce the bare base
exactly, which the last line checks rather than asserts.

In [23]:
_orig_load_base = load_base

def load_base():
    """granite-4.0-1b is a plain transformer shipped under the granitemoehybrid
    class, so transformers builds a hybrid cache it has no linear-attention
    layers for. Disabling the cache avoids that path entirely."""
    m, t = _orig_load_base()
    m.config.use_cache = False
    return m, t

In [24]:
# One load, one bank, every behavior resident. Pinning them in turn gives the
# solo numbers without reloading the model for each.
model, tok = load_base(); model.eval()

refs = {b["name"]: ref_logprobs(model, tok, DATA[b["name"]]["pairs_eval"],
                                b.get("max_len"))
        for b in BEHAVIORS if b["task"] == "dpo"}

print("bare base, nothing attached:")
baseline = {n: r[0.0]["value"]
            for n, r in eval_all(model, tok, refs, (0.0,)).items()}

bank = Bank(model, tok)
for b in BEHAVIORS:
    bank.add(b["name"], f"behaviors/{b['name']}")
print()
print("each behavior pinned in turn:")
solo = eval_all(model, tok, refs, GAMMAS, bank=bank)

print()
solo_table(solo, baseline)

Loading weights:   0%|          | 0/322 [00:00<?, ?it/s]

bare base, nothing attached:
  code      0.0: 2.881
  math      0.0: 1.374
  medical   0.0: 76.647
  summary   0.0: 13.889
  polite    0.0: 0.500

each behavior pinned in turn:
  code      0.0: 2.881  0.5: 1.910  1.0: 2.991  1.5: 5.491
  math      0.0: 1.374  0.5: 1.240  1.0: 1.455  1.5: 1.863
  medical   0.0: 76.647  0.5: 16.582  1.0: 16.399  1.5: 27.172
  summary   0.0: 13.889  0.5: 3.171  1.0: 4.252  1.5: 7.467
  polite    0.0: 0.500  0.5: 0.992  1.0: 1.000  1.5: 0.965

PPL is perplexity: lower is better.
REWARD is preference accuracy against the base: higher is better, 0.5 is chance.

behavior  metric  want         base    g=0.0    g=0.5    g=1.0    g=1.5
-----------------------------------------------------------------------
code      PPL     lower       2.881    2.881    1.910    2.991    5.491
math      PPL     lower       1.374    1.374    1.240    1.455    1.863
medical   PPL     lower      76.647   76.647   16.582   16.399   27.172
summary   PPL     lower      13.889   13.889

## 5. All of them at once

One bank, one router, every behavior resident.

Ratio is each behavior's number under routing against the same behavior
alone. Close to 1 means sharing a router costs it little.

In [25]:
# The bank from the previous cell is still loaded. Just fit the router.
bank.fit_router()
print(f"router fitted over {len(BEHAVIORS)} behaviors")

routed = {}
for mode, k in (("hard", 1), ("soft", None)):
    bank.top_k = k
    row = {}
    for b in BEHAVIORS:
        if b["task"] == "ce":
            row[b["name"]] = measure(model, tok, b, refs)["value"]
        else:
            # A style behavior is applied across domains rather than routed to,
            # which is what soft routing allows and hard routing cannot.
            with bank.pin({b["name"]: 1.0}):
                row[b["name"]] = measure(model, tok, b, refs)["value"]
    routed[mode] = row
    print(f"  {mode} routing done")

print()
print("ratio = the behavior under soft routing against the same behavior alone.")
print("1.00x means sharing a router costs it nothing.")
print()
print(f"{'behavior':<10}{'metric':<8}{'alone':>9}{'hard':>9}{'soft':>9}{'ratio':>11}")
print("-" * 56)
for b in BEHAVIORS:
    n = b["name"]; alone = solo[n][1.0]["value"]
    # perplexity improves downward and reward upward, so the ratio flips
    rec = (alone / routed["soft"][n]) if b["task"] == "ce" else (routed["soft"][n] / alone)
    tag = "" if rec <= 1.02 else "  (routing beat the behavior alone)"
    print(f"{n:<10}{solo[n][1.0]['metric']:<8}{alone:>9.3f}"
          f"{routed['hard'][n]:>9.3f}{routed['soft'][n]:>9.3f}{rec:>10.2f}x{tag}")

router fitted over 5 behaviors
  hard routing done
  soft routing done

ratio = the behavior under soft routing against the same behavior alone.
1.00x means sharing a router costs it nothing.

behavior  metric      alone     hard     soft      ratio
--------------------------------------------------------
code      PPL         2.991    2.571    1.884      1.59x  (routing beat the behavior alone)
math      PPL         1.455    1.444    1.302      1.12x  (routing beat the behavior alone)
medical   PPL        16.399   16.878   16.516      0.99x
summary   PPL         4.252    4.225    3.227      1.32x  (routing beat the behavior alone)
polite    REWARD      1.000    1.000    1.000      1.00x


## 6. Where the tokens go

Each row is one behavior's own evaluation text. Each column is a behavior. A
strong diagonal means the router discriminates, rather than flipping coins or
quietly acting as one behavior under several names.

In [26]:
names = [b["name"] for b in BEHAVIORS]


def route_weights(text):
    """Ask the bank how it splits a piece of text. Method names vary by version,
    so try the likely ones and skip the table if none is exposed."""
    for attr in ("route_weights", "routing_weights", "routes", "route",
                 "weights_for", "explain", "predict"):
        fn = getattr(bank, attr, None)
        if callable(fn):
            try:
                w = fn(text)
                if isinstance(w, dict) and w:
                    return w
            except Exception:
                continue
    return None


def eval_text(b):
    d = DATA[b["name"]]
    return [full for _, full in d["eval"]] if b["task"] == "ce" \
        else [c for _, c, _ in d["pairs_eval"]]


bank.top_k = None
if route_weights(eval_text(BEHAVIORS[0])[0][:400]) is None:
    print("this build of Bank does not expose per-text routing weights, skipping")
else:
    print(f"{'':<10}" + "".join(f"{n[:8]:>10}" for n in names))
    conf = {}
    for b in BEHAVIORS:
        acc = {n: 0.0 for n in names}
        for t in eval_text(b)[:40]:
            w = route_weights(t[:800]) or {}
            for n in names:
                acc[n] += float(w.get(n, 0.0))
        s = sum(acc.values()) or 1.0
        conf[b["name"]] = {n: acc[n] / s for n in names}
        print(f"{b['name']:<10}" + "".join(f"{conf[b['name']][n]:>10.2f}" for n in names))
    print()
    diag = sum(conf[n][n] for n in names) / len(names)
    print(f"mean self-routing {diag:.2f}; chance would be {1 / len(names):.2f}")

this build of Bank does not expose per-text routing weights, skipping


## 7. Footprint

In [27]:
n_par = sum(p.numel() for p in model.parameters())
packed = n_par * BITS / 8 / 1e6
adapters = {b["name"]: sum(os.path.getsize(os.path.join(d, f))
                           for d, _, fs in os.walk(f"behaviors/{b['name']}") for f in fs) / 1e6
            for b in BEHAVIORS}
tot = sum(adapters.values())

for n, mb in adapters.items():
    print(f"  {n:<10}{mb:>8.1f} MB")
print("-" * 22)
print(f"  {'all':<10}{tot:>8.1f} MB")
print()
print(f"base as shipped ({FAMILY}):        {packed:>9.0f} MB")
print(f"one base plus {len(BEHAVIORS)} behaviors:        {packed + tot:>9.0f} MB")
print(f"{len(BEHAVIORS)} separately fine-tuned models: {packed * len(BEHAVIORS):>9.0f} MB")
print(f"ratio: {(packed * len(BEHAVIORS)) / (packed + tot):.1f}x smaller")

bank.detach(); del model
gc.collect(); torch.cuda.empty_cache()

  code          12.8 MB
  math          12.9 MB
  medical       12.9 MB
  summary       13.0 MB
  polite         2.2 MB
----------------------
  all           53.8 MB

base as shipped (unknown):             3290 MB
one base plus 5 behaviors:             3344 MB
5 separately fine-tuned models:     16450 MB
ratio: 4.9x smaller


## Notes

Ratio near 1 means a behavior loses little by sharing a router. Well below 1
means they compete, and the routing table usually shows why.

The preference behavior is applied rather than routed to. Routing decides which
domain a token belongs to; a style is meant to apply across domains, which is
what soft routing allows.

If the base is a low-bit model, this runs its unpacked weights in PyTorch rather
than the packed model through whatever kernels ship with it. Those kernels may
round the values passing between blocks in a way PyTorch does not, which would
need checking before this ran on a device.

- https://github.com/pfekin/LARA
- https://arxiv.org/abs/2607.28669